# 특징표 관계·날짜 통제 회귀·실행 환경 확인

03 노트북에는 statsmodels가 추가로 필요합니다. 두 특징 CSV는 data/processed에 두세요. 이 코드는 원본 파형 재처리나 FFT를 수행하지 않습니다. 계수 일치는 고장 인과성을 증명하지 않습니다.


In [7]:
from pathlib import Path
import json
import sys
import importlib.metadata

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, "to_string") else value)

import os
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = Path(os.environ.get("MOTOR_FEATURE_DIR", str(PROJECT_ROOT / "data" / "processed")))
OUT = PROJECT_ROOT / "results" / "reproduction"
OUT.mkdir(parents=True, exist_ok=True)

clean = pd.read_csv(
    DATA / "ai_ready_time_features_clean.csv",
    encoding="utf-8-sig",
)

if clean["path"].duplicated().any():
    raise ValueError("clean CSV에 중복 원본 경로가 있습니다.")

raw_features_path = DATA / "ai_ready_time_features.csv"

if raw_features_path.exists():
    before = pd.read_csv(raw_features_path, encoding="utf-8-sig")

    if before["path"].duplicated().any():
        raise ValueError("필터 전 CSV에 중복 원본 경로가 있습니다.")

    if not clean["path"].isin(before["path"]).all():
        raise ValueError("clean CSV에 필터 전 CSV에 없는 경로가 있습니다.")

    removed = before.loc[~before["path"].isin(clean["path"])].copy()
    retained = (
        before.loc[before["path"].isin(clean["path"])]
        .sort_values("path")
        .reset_index(drop=True)
    )
    clean_sorted = clean.sort_values("path").reset_index(drop=True)

    if not retained.equals(clean_sorted):
        raise ValueError("유지된 행의 특징 값이 두 CSV에서 다릅니다.")

    removed["mean_rms"] = removed[["R_rms", "S_rms", "T_rms"]].mean(axis=1)
    display(removed[["equipment", "label", "mean_rms"]])
    print("제거 수:", len(removed))

    removed.to_csv(
        OUT / "removed_feature_rows.csv",
        index=False,
        encoding="utf-8-sig",
    )
else:
    print("필터 전 CSV가 없어 차분 검증을 건너뜁니다.")

date_tokens = clean["path"].str.extract(
    r"_(\d{8})_\d{6}_",
    expand=False,
)
if date_tokens.isna().any():
    raise ValueError("날짜를 추출할 수 없는 파일명이 있습니다.")

clean["day"] = pd.to_datetime(
    date_tokens,
    format="%Y%m%d",
    errors="raise",
).dt.date

x = clean.loc[clean["equipment"].eq("L-DSF-01")].copy()
t = pd.crosstab(x["day"], x["label"])
overlap = t.index[(t["정상"] > 0) & (t["축정렬불량"] > 0)]

x = x.loc[x["day"].isin(overlap)].copy()
x["fault"] = x["label"].eq("축정렬불량").astype(int)

print("검증 날짜 수:", len(overlap))
print("검증 샘플 수:", len(x))

reference = {
    "OLS": [
        0.1327657279313804,
        0.10981770491910964,
        0.12596792470453333,
    ],
    "Quantile": [
        0.08477298440467462,
        0.06745446364050878,
        0.07369081090284979,
    ],
}

rows = []

for i, col in enumerate(["R_rms", "S_rms", "T_rms"]):
    for method in ["OLS", "Quantile"]:
        formula = f"{col} ~ fault + C(day)"

        if method == "OLS":
            model = smf.ols(formula, data=x).fit()
        else:
            model = smf.quantreg(formula, data=x).fit(q=0.5)

        actual = float(model.params["fault"])
        expected = reference[method][i]

        rows.append({
            "method": method,
            "feature": col,
            "coef": actual,
            "original_coef": expected,
            "abs_difference": abs(actual - expected),
            "close_at_1e-6": bool(
                np.isclose(actual, expected, rtol=0, atol=1e-6)
            ),
            "p": float(model.pvalues["fault"]),
        })

result = pd.DataFrame(rows)
display(result)

result.to_csv(
    OUT / "regression_verification.csv",
    index=False,
    encoding="utf-8-sig",
)

versions = {
    "python": sys.version,
    "executable": sys.executable,
}

for package in [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "statsmodels",
    "xgboost",
    "lightgbm",
]:
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = "not installed"

print(json.dumps(versions, indent=2, ensure_ascii=False))

(OUT / "environment.json").write_text(
    json.dumps(versions, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("결과 저장 위치:", OUT)

,equipment,label,mean_rms
1002,L-DSF-01,정상,0.118691
1004,L-DSF-01,정상,0.145194
1006,L-DSF-01,정상,0.118707
1023,L-DSF-01,정상,0.119432
1027,L-DSF-01,정상,0.119878
1028,L-DSF-01,정상,0.118786
1048,L-DSF-01,정상,0.120815


제거 수: 7
검증 날짜 수: 9
검증 샘플 수: 407


,method,feature,coef,original_coef,abs_difference,close_at_1e-6,p
0,OLS,R_rms,0.132766,0.132766,9.992007e-16,True,1.123072e-19
1,Quantile,R_rms,0.084773,0.084773,2.491213e-10,True,2.301505e-97
2,OLS,S_rms,0.109818,0.109818,3.608225e-16,True,1.426609e-16
3,Quantile,S_rms,0.067454,0.067454,1.784954e-10,True,1.871673e-80
4,OLS,T_rms,0.125968,0.125968,1.387779e-16,True,1.721446e-14
5,Quantile,T_rms,0.073691,0.073691,1.598721e-13,True,1.421811e-46


{
  "python": "3.11.17 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:16:54) [MSC v.1942 64 bit (AMD64)]",
  "executable": "C:\\Users\\USER\\anaconda3\\envs\\motor-check\\python.exe",
  "numpy": "2.4.6",
  "pandas": "3.0.6",
  "scipy": "1.17.1",
  "scikit-learn": "1.9.1",
  "statsmodels": "0.15.0",
  "xgboost": "3.2.0",
  "lightgbm": "4.7.0"
}
